In [4]:
import numpy as np

def read_contcar(filename):
    """Read CONTCAR file and extract lattice parameters and atomic positions"""
    with open(filename, 'r') as f:
        lines = f.readlines()
    
    # Parse header information
    comment = lines[0].strip()
    scale_factor = float(lines[1].strip())
    
    # Read lattice vectors (already scaled by scale_factor in the file)
    lattice = np.array([
        [float(x) for x in lines[2].split()],
        [float(x) for x in lines[3].split()],
        [float(x) for x in lines[4].split()]
    ])
    
    # Read element types and counts
    elements = lines[5].split()
    element_counts = [int(x) for x in lines[6].split()]
    
    # Check if Direct or Cartesian coordinates
    coord_type = lines[7].strip()
    
    # Read atomic positions
    positions = []
    start_line = 8
    total_atoms = sum(element_counts)
    
    for i in range(start_line, start_line + total_atoms):
        pos = [float(x) for x in lines[i].split()[:3]]
        positions.append(pos)
    
    positions = np.array(positions)
    
    # Convert to direct coordinates if currently Cartesian
    if coord_type.lower().startswith('c'):  # Cartesian
        positions = cartesian_to_direct(positions, lattice)
    
    return {
        'comment': comment,
        'scale_factor': scale_factor,
        'lattice': lattice,
        'elements': elements,
        'element_counts': element_counts,
        'coord_type': coord_type,
        'positions': positions
    }

def direct_to_cartesian(direct_coords, lattice):
    """Convert direct coordinates to Cartesian coordinates"""
    return np.dot(direct_coords, lattice)

def cartesian_to_direct(cartesian_coords, lattice):
    """Convert Cartesian coordinates to direct coordinates"""
    inv_lattice = np.linalg.inv(lattice)
    return np.dot(cartesian_coords, inv_lattice)

def read_xdatcar(filename, skip_configs=5000, end_config=60000):
    """Read XDATCAR file and extract configurations from skip_configs+1 to end_config"""
    with open(filename, 'r') as f:
        lines = f.readlines()
    
    # Parse header (similar to CONTCAR but with different format)
    comment = lines[0].strip()
    scale_factor = float(lines[1].strip())
    
    # Read lattice vectors
    lattice = np.array([
        [float(x) for x in lines[2].split()],
        [float(x) for x in lines[3].split()],
        [float(x) for x in lines[4].split()]
    ]) * scale_factor
    
    # Read element types and counts
    elements = lines[5].split()
    element_counts = [int(x) for x in lines[6].split()]
    total_atoms = sum(element_counts)
    
    configurations = []
    current_line = 7  # Start after header
    config_count = 0
    
    while current_line < len(lines):
        if lines[current_line].strip().startswith('Direct configuration='):
            config_count += 1
            current_line += 1
            
            # Only process configurations after skip_configs and before end_config
            if skip_configs < config_count <= end_config:
                config_positions = []
                for i in range(total_atoms):
                    if current_line + i < len(lines):
                        pos = [float(x) for x in lines[current_line + i].split()[:3]]
                        config_positions.append(pos)
                
                configurations.append(np.array(config_positions))
            
            current_line += total_atoms
        else:
            current_line += 1
    
    return {
        'lattice': lattice,
        'elements': elements,
        'element_counts': element_counts,
        'configurations': configurations
    }

def calculate_com(positions, masses):
    """Calculate center of mass for given positions and masses"""
    total_mass = np.sum(masses)
    com = np.sum(positions * masses.reshape(-1, 1), axis=0) / total_mass
    return com

def identify_framework_atoms(elements, element_counts):
    """
    Identify framework atoms (T-sites: Si, Al and O atoms)
    Returns the number of framework atoms
    """
    framework_elements = {'Si', 'Al', 'O'}
    framework_atom_count = 0
    
    for i, element in enumerate(elements):
        if element in framework_elements:
            framework_atom_count += element_counts[i]
    
    return framework_atom_count

def get_guest_atom_info(elements, element_counts, framework_atom_count):
    """
    Get information about guest atoms (non-framework atoms)
    Returns guest elements and their counts
    """
    guest_elements = []
    guest_counts = []
    atom_count = 0
    
    for i, (element, count) in enumerate(zip(elements, element_counts)):
        if atom_count >= framework_atom_count:
            # This element is entirely guest atoms
            guest_elements.append(element)
            guest_counts.append(count)
        elif atom_count + count > framework_atom_count:
            # This element spans framework and guest atoms
            guest_count = atom_count + count - framework_atom_count
            guest_elements.append(element)
            guest_counts.append(guest_count)
        atom_count += count
    
    return guest_elements, guest_counts
    """Return atomic masses for common elements (in atomic mass units)"""
    masses = {
        'H': 1.008, 'He': 4.003, 'Li': 6.941, 'Be': 9.012, 'B': 10.811,
        'C': 12.011, 'N': 14.007, 'O': 15.999, 'F': 18.998, 'Ne': 20.180,
        'Na': 22.990, 'Mg': 24.305, 'Al': 26.982, 'Si': 28.086, 'P': 30.974,
        'S': 32.065, 'Cl': 35.453, 'Ar': 39.948, 'K': 39.098, 'Ca': 40.078,
        'Sc': 44.956, 'Ti': 47.867, 'V': 50.942, 'Cr': 51.996, 'Mn': 54.938,
        'Fe': 55.845, 'Co': 58.933, 'Ni': 58.693, 'Cu': 63.546, 'Zn': 65.38
    }
    return masses

def process_files(contcar_file='CONTCAR', xdatcar_file='XDATCAR', output_file='POSCAR'):
    """Main function to process CONTCAR and XDATCAR files and generate POSCAR"""
    
    # Step 1: Read CONTCAR
    print("Reading CONTCAR...")
    contcar_data = read_contcar(contcar_file)
    
    # Identify framework atoms (T-sites: Si, Al + O atoms)
    framework_atom_count = identify_framework_atoms(contcar_data['elements'], contcar_data['element_counts'])
    print(f"Framework atoms identified: {framework_atom_count}")
    print(f"Framework composition: {dict(zip(contcar_data['elements'][:2], contcar_data['element_counts'][:2]))}")
    
    # Step 2: Read XDATCAR
    print("Reading XDATCAR...")
    xdatcar_data = read_xdatcar(xdatcar_file, skip_configs=5000, end_config=60000)
    
    # Get atomic masses
    atomic_masses = get_atomic_masses()
    
    # Identify guest atoms in XDATCAR
    xdatcar_framework_atoms = identify_framework_atoms(xdatcar_data['elements'], xdatcar_data['element_counts'])
    guest_elements, guest_counts = get_guest_atom_info(
        xdatcar_data['elements'], xdatcar_data['element_counts'], xdatcar_framework_atoms
    )
    
    print(f"Guest atoms in XDATCAR: {dict(zip(guest_elements, guest_counts))}")
    
    # Step 3: Process configurations and calculate COM
    print("Processing configurations and calculating COM...")
    all_com_positions = []
    
    for config_idx, config in enumerate(xdatcar_data['configurations']):
        if config_idx % 1000 == 0:
            print(f"Processing configuration {config_idx + 5001}...")
        
        # Extract positions for guest atoms (atoms after framework atoms)
        guest_positions_direct = config[xdatcar_framework_atoms:]
        
        # Convert to Cartesian coordinates
        guest_positions_cartesian = direct_to_cartesian(guest_positions_direct, xdatcar_data['lattice'])
        
        # Create mass array for guest atoms
        guest_masses = []
        for elem, count in zip(guest_elements, guest_counts):
            mass = atomic_masses.get(elem, 1.0)  # Default to 1.0 if element not found
            guest_masses.extend([mass] * count)
        
        guest_masses = np.array(guest_masses[:len(guest_positions_cartesian)])
        
        # Calculate COM in Cartesian coordinates
        if len(guest_positions_cartesian) > 0 and len(guest_masses) > 0:
            com_cartesian = calculate_com(guest_positions_cartesian, guest_masses)
            
            # Convert COM back to direct coordinates
            com_direct = cartesian_to_direct(com_cartesian.reshape(1, -1), contcar_data['lattice'])[0]
            all_com_positions.append(com_direct)
    
    # Step 4: Prepare output with only framework atoms and COM positions
    print("Writing POSCAR...")
    
    # Get only framework atoms from CONTCAR
    contcar_framework_atoms = identify_framework_atoms(contcar_data['elements'], contcar_data['element_counts'])
    framework_positions = contcar_data['positions'][:contcar_framework_atoms]
    
    # Get framework elements and counts only
    framework_elements = []
    framework_element_counts = []
    for i, element in enumerate(contcar_data['elements']):
        if element in {'Si', 'Al', 'O'}:
            framework_elements.append(element)
            framework_element_counts.append(contcar_data['element_counts'][i])
    
    # Prepare the output structure (framework + COM points)
    output_positions = np.vstack([framework_positions, np.array(all_com_positions)])
    
    # Update element information to include COM points
    output_elements = framework_elements + ['COM']
    output_element_counts = framework_element_counts + [len(all_com_positions)]
    
    # Write POSCAR file
    with open(output_file, 'w') as f:
        f.write(f"{contcar_data['comment']} + COM points\n")
        f.write(f"   {contcar_data['scale_factor']:.14f}\n")
        
        # Write lattice vectors
        for row in contcar_data['lattice']:
            f.write(f"    {row[0]:20.16f} {row[1]:20.16f} {row[2]:20.16f}\n")
        
        # Write element names and counts
        f.write("   " + "    ".join(output_elements) + "\n")
        f.write("   " + "    ".join(map(str, output_element_counts)) + "\n")
        
        # Write coordinate type (use Direct for consistency)
        f.write("Direct\n")
        
        # Write all positions
        for pos in output_positions:
            f.write(f"  {pos[0]:18.16f}  {pos[1]:18.16f}  {pos[2]:18.16f}\n")
    
    print(f"Successfully created {output_file} with {len(all_com_positions)} COM positions")
    print(f"Total atoms in output: {len(output_positions)}")
    print(f"Framework atoms: {contcar_framework_atoms}")
    print(f"COM points added: {len(all_com_positions)}")
    print(f"Guest atoms from CONTCAR excluded: {sum(contcar_data['element_counts']) - contcar_framework_atoms}")

# Example usage
if __name__ == "__main__":
    process_files('CONTCAR', 'XDATCAR', 'POSCAR')

Reading CONTCAR...
Framework atoms identified: 36
Framework composition: {'Si': 11, 'O': 24}
Reading XDATCAR...
Guest atoms in XDATCAR: {'Cu': 1}
Processing configurations and calculating COM...
Processing configuration 5001...
Processing configuration 6001...
Processing configuration 7001...
Processing configuration 8001...
Processing configuration 9001...
Processing configuration 10001...
Processing configuration 11001...
Processing configuration 12001...
Processing configuration 13001...
Processing configuration 14001...
Processing configuration 15001...
Processing configuration 16001...
Processing configuration 17001...
Processing configuration 18001...
Processing configuration 19001...
Processing configuration 20001...
Processing configuration 21001...
Processing configuration 22001...
Processing configuration 23001...
Processing configuration 24001...
Processing configuration 25001...
Processing configuration 26001...
Processing configuration 27001...
Processing configuration 280